# FedGRAA — Real-Data Reproduction on Google Colab

Runs the **real** experiment on CIC-IDS-2017 (and optionally UNSW-NB15):

```
python -m fedgraa gamma-sweep --dataset CIC-IDS-2017 --data-dir <your CSVs> --rounds 200
```

The implementation is pure NumPy (no TensorFlow needed), so a standard **CPU** runtime is fine — no GPU required. 200 rounds × 14 clients is CPU-bound; expect a while. Start with `--rounds 60` to smoke-test, then go to 200.

**You need two things:** the `fedgraa` package (this repo) and the CIC-IDS-2017 CSV files. The steps below get both.

## Step 1 — Get the FedGRAA code into Colab

Pick **one** path by setting `USE_DRIVE_FOR_CODE`:
* **Upload a zip** (default) — on your laptop, zip the `PAKDD` folder (the one containing the `fedgraa/` package) and upload it here.
* **Google Drive** — if the repo already lives in your Drive, set the flag to `True` and edit `REPO_DIR`.

In [ ]:
USE_DRIVE_FOR_CODE = False  # True if your PAKDD repo is already in Google Drive

import os, sys

if USE_DRIVE_FOR_CODE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/Research_Project/PAKDD'  # <-- EDIT to your path
else:
    from google.colab import files
    import zipfile, io
    print('Upload a .zip of your PAKDD folder (must contain the fedgraa/ package)...')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
        z.extractall('/content/fedgraa_repo')
    # Locate the directory that actually contains the fedgraa package.
    REPO_DIR = None
    for root, dirs, _ in os.walk('/content/fedgraa_repo'):
        if os.path.exists(os.path.join(root, 'fedgraa', '__init__.py')):
            REPO_DIR = root
            break
    assert REPO_DIR, 'Could not find the fedgraa package inside the uploaded zip.'

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('REPO_DIR =', REPO_DIR)
print('contents:', sorted(os.listdir(REPO_DIR))[:20])

In [ ]:
# Dependencies (Colab already ships numpy / pandas / scikit-learn; this is a no-op safety net).
!pip install -q "numpy>=1.24" "pandas>=2.0" "scikit-learn>=1.3" matplotlib

import fedgraa
from fedgraa.config import CIC_IDS_2017, UNSW_NB15
print('fedgraa version:', fedgraa.__version__)
print('CIC-IDS-2017:', CIC_IDS_2017.n_clients, 'clients,', CIC_IDS_2017.n_classes, 'classes')
print('UNSW-NB15  :', UNSW_NB15.n_clients, 'clients,', UNSW_NB15.n_classes, 'classes')

## Step 2 — Get the real CIC-IDS-2017 CSVs

The loader reads **every `.csv` in a folder**, concatenates them, cleans `inf`/`NaN`, maps the real label spellings onto the paper's classes (including `Web Attack – Brute Force → Brute Force`), and caps frequent classes at 10,000.

Pick **one** source below and set `DATA_DIR` to the folder holding the CSVs.

### Option A — Google Drive (most reliable)
Upload the CIC-IDS-2017 `MachineLearningCVE` CSVs to a Drive folder, then mount and point `DATA_DIR` at it.

### Option B — Kaggle API
Upload your `kaggle.json` token and download a CICIDS-2017 dataset. Replace the slug with the dataset you choose (e.g. one that contains the 8 `MachineLearningCVE` CSVs).

In [ ]:
# === Option A: Google Drive ===
MOUNT_DRIVE_FOR_DATA = True
if MOUNT_DRIVE_FOR_DATA:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print('drive already mounted or:', e)

# EDIT this to the folder that holds your CIC-IDS-2017 *.csv files:
DATA_DIR = '/content/drive/MyDrive/datasets/CIC-IDS-2017'

In [ ]:
# === Option B: Kaggle API (uncomment to use instead of Drive) ===
from google.colab import files
print('Upload your kaggle.json API token...')
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle

# Replace <owner>/<dataset-slug> with a CICIDS-2017 dataset that contains the
# MachineLearningCVE CSVs. Inspect the extracted folder and set DATA_DIR to it.
!kaggle datasets download -d <owner>/<dataset-slug> -p /content/cicids --unzip
DATA_DIR = '/content/cicids'  # adjust to the subfolder that actually holds the .csv files

In [ ]:
# Sanity-check the data folder before the (slow) training run.
import glob, os
import pandas as pd

csvs = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print('DATA_DIR =', DATA_DIR)
print('found', len(csvs), 'CSV files:')
for c in csvs:
    print('  -', os.path.basename(c))
assert csvs, f'No CSV files found in {DATA_DIR}. Fix DATA_DIR above.'

# Peek at the label column distribution of the first file.
head = pd.read_csv(csvs[0], nrows=20000)
head.columns = [c.strip() for c in head.columns]
label_col = next((c for c in head.columns if c.strip().lower() in ('label', 'attack', 'attack_cat')), head.columns[-1])
print('\nlabel column:', label_col)
print(head[label_col].value_counts().head(20))

## Step 3 — Run the real experiment

This is the exact command you asked for:

```
python -m fedgraa gamma-sweep --dataset CIC-IDS-2017 --data-dir <DATA_DIR> --rounds 200
```

It sweeps γ ∈ {0, 0.5, 1} and reports overall accuracy and the **last-n rare-class accuracy** (the metric the paper improves).

> **Smoke-test first** with 60 rounds (next cell). Once that runs end-to-end, launch the full 200-round cell.

In [ ]:
# Smoke test: 60 rounds, print metrics every 20 rounds.
!python -m fedgraa gamma-sweep --dataset CIC-IDS-2017 --data-dir "{DATA_DIR}" --rounds 60 --eval-every 20

In [ ]:
# THE REAL DEAL: full 200-round gamma sweep (this is the slow one).
!python -m fedgraa gamma-sweep --dataset CIC-IDS-2017 --data-dir "{DATA_DIR}" --rounds 200

## Step 4 — Run programmatically and plot

Same sweep, but called from Python so you can capture the numbers and chart them. Use this if you want figures instead of just stdout.

In [ ]:
from fedgraa.experiment import run_gamma_sweep, summarize_gamma_sweep

# Use rounds=200 to match the paper; lower it while iterating.
results = run_gamma_sweep(
    dataset='CIC-IDS-2017',
    data_dir=DATA_DIR,
    rounds=200,
    gammas=(0.0, 0.5, 1.0),
    n_rare=3,
    seed=0,
)
print(summarize_gamma_sweep(results))

In [ ]:
import matplotlib.pyplot as plt

# Plot last-n rare-class accuracy vs communication round for each method.
plot_order = [
    ('fedavg',        'FedAvg'),
    ('static',        'Static Reparam.'),
    ('fedgraa_g0.0',  'FedGRAA (g=0)'),
    ('fedgraa_g0.5',  'FedGRAA (g=0.5)'),
    ('fedgraa_g1.0',  'FedGRAA (g=1)'),
]

plt.figure(figsize=(8, 5))
for key, label in plot_order:
    if key in results:
        h = results[key].history
        plt.plot(h.rounds, [a * 100 for a in h.last_n_acc], marker='o', label=label)
plt.xlabel('Communication round')
plt.ylabel('Last-n rare-class accuracy (%)')
plt.title('CIC-IDS-2017 — rare-class accuracy (Figure 1 style)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5 — Optional extras

* **n-sweep** (Figures 2-3): FedGRAA robustness as the number of protected rare clients grows, n ∈ {3, 5, 7}.
* **UNSW-NB15**: re-run the whole thing on the second dataset by pointing `--data-dir` at its CSVs.

In [ ]:
# Figures 2-3: FedGRAA(g=1) vs baselines for n in {3, 5, 7}.
!python -m fedgraa n-sweep --dataset CIC-IDS-2017 --data-dir "{DATA_DIR}" --gamma 1 --rounds 200

In [ ]:
# Optional: the same gamma sweep on UNSW-NB15.
# Set UNSW_DATA_DIR to the folder holding the UNSW-NB15 CSVs, then run.
# UNSW_DATA_DIR = '/content/drive/MyDrive/datasets/UNSW-NB15'
# !python -m fedgraa gamma-sweep --dataset UNSW-NB15 --data-dir "{UNSW_DATA_DIR}" --rounds 200